# 01 - EDA & Preprocessing (Multi-Config)
## CICDDoS2019

**MODE=0:** Local (5K rows) | **MODE=1:** SageMaker (10K, 30K, 50K)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob, os, pickle, gc, warnings
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')

# ===== MODE SWITCH (read from mode.txt) =====
# Edit mode.txt: 0 = Local (5K), 1 = SageMaker (10K, 30K, 50K)
MODE = int(open('mode.txt').read().strip())
# ==============================================

if MODE == 0:
    CONFIGS = [5000]
elif MODE == 1:
    CONFIGS = [10000, 30000, 50000]
    # Auto-download from S3 if data not present
    if not os.path.exists('../data/CIC-DDoS2019-Dataset/01-12/'):
        print('Downloading dataset from S3...')
        os.system('aws s3 sync s3://ssh-detection-features-232032302717/datasets/CICDDoS2019/ ../data/CIC-DDoS2019-Dataset/ --quiet')
        print('Download complete.')

RANDOM_SEED = 42
TRAIN_DIR = '../data/CIC-DDoS2019-Dataset/01-12/'
TEST_DIR = '../data/CIC-DDoS2019-Dataset/03-11/'
OUTPUT_DIR = '../data/'

np.random.seed(RANDOM_SEED)
print(f'MODE={MODE} | Configs: {CONFIGS}')

In [ ]:
def load_and_sample(filepath, max_rows, seed=42):
    chunks = []
    total_read = 0
    try:
        for chunk in pd.read_csv(filepath, chunksize=50000, low_memory=False,
                                  encoding='utf-8', on_bad_lines='skip'):
            chunks.append(chunk)
            total_read += len(chunk)
            if total_read >= max_rows * 3:
                break
    except Exception as e:
        print(f'  Error: {e}')
        return None
    df = pd.concat(chunks, ignore_index=True)
    df.columns = df.columns.str.strip()
    label_col = [c for c in df.columns if 'label' in c.lower()]
    LABEL = label_col[0] if label_col else df.columns[-1]
    if len(df) > max_rows:
        sampled_dfs = []
        for label_val, group in df.groupby(LABEL):
            proportion = len(group) / len(df)
            n_sample = max(1, int(max_rows * proportion))
            n_sample = min(n_sample, len(group))
            sampled_dfs.append(group.sample(n=n_sample, random_state=seed))
        df = pd.concat(sampled_dfs, ignore_index=True)
    del chunks; gc.collect()
    return df

def clean_dataset(df, label_col):
    ts_cols = [c for c in df.columns if 'timestamp' in c.lower()]
    if ts_cols:
        df = df.drop(columns=ts_cols)
    y = df[label_col].copy()
    X = df.drop(columns=[label_col]).select_dtypes(include=[np.number])
    X = X.replace([np.inf, -np.inf], np.nan)
    mask = ~X.isnull().any(axis=1)
    return X[mask].reset_index(drop=True), y[mask].reset_index(drop=True)

def standardize(label):
    label = str(label).strip()
    if label.startswith('DrDoS_'):
        label = label.replace('DrDoS_', '')
    return label.upper()

print('Functions OK')

In [ ]:
train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, '*.csv')))
test_files = sorted(glob.glob(os.path.join(TEST_DIR, '*.csv')))
print(f'Training files: {len(train_files)} | Testing files: {len(test_files)}')

for config in CONFIGS:
    suffix = f'{config//1000}K'
    print(f'\n{"="*70}')
    print(f'  PROCESSING: {suffix} rows per file')
    print(f'{"="*70}')
    
    # Training
    train_dfs = []
    for f in train_files:
        r = load_and_sample(f, max_rows=config, seed=RANDOM_SEED)
        if r is not None: train_dfs.append(r)
    df_train = pd.concat(train_dfs, ignore_index=True)
    df_train.columns = df_train.columns.str.strip()
    del train_dfs; gc.collect()
    
    # Testing
    test_dfs = []
    for f in test_files:
        r = load_and_sample(f, max_rows=config, seed=RANDOM_SEED)
        if r is not None: test_dfs.append(r)
    df_test = pd.concat(test_dfs, ignore_index=True)
    df_test.columns = df_test.columns.str.strip()
    del test_dfs; gc.collect()
    
    # Clean
    label_col = [c for c in df_train.columns if 'label' in c.lower()]
    LABEL = label_col[0] if label_col else df_train.columns[-1]
    X_train, y_train = clean_dataset(df_train, LABEL)
    X_test, y_test = clean_dataset(df_test, LABEL)
    del df_train, df_test; gc.collect()
    
    common = sorted(set(X_train.columns) & set(X_test.columns))
    X_train = X_train[common]
    X_test = X_test[common]
    
    # Encode
    y_train_std = y_train.apply(standardize)
    y_test_std = y_test.apply(standardize)
    le = LabelEncoder()
    all_labels = sorted(set(y_train_std.unique()) | set(y_test_std.unique()))
    le.fit(all_labels)
    y_train_enc = le.transform(y_train_std)
    y_test_enc = le.transform(y_test_std)
    
    # Save
    train_data = {'X': X_train, 'y': y_train_enc, 'feature_names': list(X_train.columns),
                  'label_encoder': le, 'classes': list(le.classes_), 'config': config}
    test_data = {'X': X_test, 'y': y_test_enc, 'feature_names': list(X_test.columns),
                 'label_encoder': le, 'classes': list(le.classes_), 'config': config}
    with open(os.path.join(OUTPUT_DIR, f'cleaned_train_{suffix}.pkl'), 'wb') as f:
        pickle.dump(train_data, f)
    with open(os.path.join(OUTPUT_DIR, f'cleaned_test_{suffix}.pkl'), 'wb') as f:
        pickle.dump(test_data, f)
    
    print(f'  Train: {len(X_train):,} samples, {X_train.shape[1]} features')
    print(f'  Test:  {len(X_test):,} samples')
    print(f'  Classes: {len(le.classes_)}')
    del X_train, X_test; gc.collect()

print(f'\n=== DONE (MODE={MODE}). Proceed to notebook 02. ===')